In [ ]:
import importlib
import numpy as np
import pandas as pd

import load_data
import assign_labels
import build_features
import extract_ontology
import build_topology_graph
import build_ontology

importlib.reload(load_data)
importlib.reload(assign_labels)
importlib.reload(build_features)
importlib.reload(extract_ontology)
importlib.reload(build_topology_graph)
importlib.reload(build_ontology)

from load_data import load_alerts_from_json
from assign_labels import add_labels_to_df
from extract_ontology import topology_edges, host_sets
from build_topology_graph import plot_scenario_topologies
from build_ontology import build_all_scenarios

### Load data
##### Wazuh + Aminer Alerts (AIT-ADS, .json format) to .csv

In [ ]:
output_file = "combined_ait.csv"
dir_path = "../data/ait_ads"

df = load_alerts_from_json(output_file, dir_path)

Opening file ../data/ait_ads/harrison_aminer.json...
Opening file ../data/ait_ads/wardbeck_wazuh.json...
Opening file ../data/ait_ads/wheeler_wazuh.json...
Opening file ../data/ait_ads/shaw_wazuh.json...
Opening file ../data/ait_ads/wilson_aminer.json...
Opening file ../data/ait_ads/fox_aminer.json...
Opening file ../data/ait_ads/santos_wazuh.json...
Opening file ../data/ait_ads/fox_wazuh.json...
Opening file ../data/ait_ads/shaw_aminer.json...
Opening file ../data/ait_ads/wheeler_aminer.json...
Opening file ../data/ait_ads/russellmitchell_aminer.json...
Opening file ../data/ait_ads/harrison_wazuh.json...
Opening file ../data/ait_ads/wilson_wazuh.json...
Opening file ../data/ait_ads/santos_aminer.json...
Opening file ../data/ait_ads/russellmitchell_wazuh.json...
Opening file ../data/ait_ads/wardbeck_aminer.json...
Writing data to combined_ait.csv...

...
Done.


: 

##### Load .csv to notebook

In [ ]:
df = pd.read_csv("../data/ait_ads/combined_ait.csv")
df.head()
print(df["timestamp"].min(), df["timestamp"].max())
df["timestamp"].map(type).value_counts()

/var/folders/69/6nn2sm8d7qq0p98t_qgp1ckr0000gn/T/ipykernel_3174/912057347.py:1: DtypeWarning: Columns (7,10,12,13,14,17,24,31,32,36,37,38,39,40,41,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/ait_ads/combined_ait.csv")


2022-01-14 00:00:01.750000+00:00 2022-02-08 23:59:46.130000114+00:00


timestamp
<class 'str'>    2655821
Name: count, dtype: int64

: 

#### Sanity checks

In [ ]:
# number of attakcs in the dataset for each scenario
print(df["y"].value_counts())

df[df["y"] == 1]["scenario"].value_counts()
# or
df.groupby("scenario")["y"].sum().sort_values(ascending=False)


#### Topology + ontology

In [ ]:
edge_table = topology_edges(df, min_weight=3)
edge_table.head(10)

,scenario,srcip,dstip,proto,dstport,weight
0,fox,10.35.33.111,91.189.88.142,TCP,80.0,28
1,fox,10.35.33.111,91.189.88.152,TCP,80.0,16
2,fox,10.35.33.111,91.189.91.38,TCP,80.0,6
4,fox,10.35.33.111,91.189.95.85,TCP,80.0,10
5,fox,10.35.35.206,91.189.88.142,TCP,80.0,16
6,fox,10.35.35.206,91.189.88.152,TCP,80.0,24
8,fox,10.35.35.206,91.189.91.39,TCP,80.0,8
9,fox,10.35.35.206,91.189.95.85,TCP,80.0,10
10,fox,100.21.15.207,192.168.128.4,TCP,49634.0,4
11,fox,100.24.82.208,192.168.128.4,TCP,58058.0,4


: 

In [ ]:
host_summary = host_sets(df, edge_table)
host_summary

/Users/annavisman/stack/TUDelft/thesis/msc-thesis/aact/extract_ontology.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: set(g["srcip"]).union(set(g["dstip"])))


,scenario,n_active_hosts,n_hosts_in_edges,isolated_but_active
0,fox,13,889,6
1,harrison,11,951,3
2,russellmitchell,11,634,3
3,santos,11,1212,3
4,shaw,12,1094,10
5,wardbeck,12,1409,4
6,wheeler,13,1448,5
7,wilson,11,1461,3


: 

In [ ]:
scenarios = sorted(edge_table["scenario"].unique())

for sc in scenarios:
    plot_scenario_topologies(edge_table, sc, out_dir="../out/topologies/", collapse=True, min_weight=10, remove_hubs_k=3, make_subnet_view=True, subnet_k=3)

: 

In [ ]:
build_all_scenarios("../out/ontology_inputs/scenario_hosts.csv",
                    "../out/ontology_inputs/scenario_services.csv",
                    "../out/ontology_inputs/scenario_edges.csv",
                    "../data/ait_ads/combined_ait.csv",
                    out_dir="../out/ontology_out",
                    k=3)

/Users/annavisman/stack/TUDelft/thesis/msc-thesis/aact/build_ontology.py:156: DtypeWarning: Columns (7,10,11,12,15,22,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  alerts_df = pd.read_csv(alerts_csv)


Wrote ../out/ontology_out/ontology_fox.ttl
Wrote ../out/ontology_out/ontology_harrison.ttl
Wrote ../out/ontology_out/ontology_russellmitchell.ttl
Wrote ../out/ontology_out/ontology_santos.ttl
Wrote ../out/ontology_out/ontology_shaw.ttl
Wrote ../out/ontology_out/ontology_wardbeck.ttl
Wrote ../out/ontology_out/ontology_wheeler.ttl
Wrote ../out/ontology_out/ontology_wilson.ttl


: 

## Labelling

In [ ]:
add_labels_to_df(df)

event_label
benign    2509439
attack     146382
Name: count, dtype: int64
scenario         event_label
fox              benign         441601
                 attack          31503
harrison         benign         561805
                 attack          32143
russellmitchell  benign          39089
                 attack           6455
santos           benign         123766
                 attack           7013
shaw             benign          69009
                 attack           1773
wardbeck         benign          89296
                 attack           1961
wheeler          benign         580288
                 attack          35873
wilson           benign         604585
                 attack          29661
Name: count, dtype: int64
Writing to output file...
Done.


: 

In [ ]:
df.columns

Index(['timestamp', 'category', 'entity', 'raw_log', 'scenario', 'source',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'srcip', 'dstip', 'srcport', 'dstport', 'proto',
       'is_auth_event', 'is_cred_event', 'is_web_event', 'is_cron',
       'is_success', 'is_uid0', 'aminer_component_type',
       'aminer_training_mode', 'aminer_new_event', 'wazuh_level',
       'wazuh_antivirus', 'wazuh_update', 'is_ids_alert', 'ids_signature',
       'ids_category', 'ids_severity', 'exploit_class', 'alert_channel',
       'decoder', 'decoder_parent', 'location', 'mitre_ids', 'mitre_tactic',
       'mitre_technique', 'username', 'procname'],
      dtype='object')

: 

In [ ]:
df[df["rule_desc"]=="IDS event."][["ids_signature","ids_category","ids_severity","proto","srcip","dstip","dstport","exploit_class"]].head(5)

,ids_signature,ids_category,ids_severity,proto,srcip,dstip,dstport,exploit_class
10204,ET POLICY GNU/Linux APT User-Agent Outbound li...,Not Suspicious Traffic,3.0,TCP,192.168.96.4,91.189.95.85,80.0,policy_indicator
10206,ET POLICY GNU/Linux APT User-Agent Outbound li...,Not Suspicious Traffic,3.0,TCP,192.168.96.4,91.189.88.152,80.0,policy_indicator
10207,ET POLICY GNU/Linux APT User-Agent Outbound li...,Not Suspicious Traffic,3.0,TCP,192.168.96.4,91.189.88.152,80.0,policy_indicator
10210,ET POLICY GNU/Linux APT User-Agent Outbound li...,Not Suspicious Traffic,3.0,TCP,192.168.96.4,91.189.88.152,80.0,policy_indicator
10211,ET POLICY GNU/Linux APT User-Agent Outbound li...,Not Suspicious Traffic,3.0,TCP,192.168.96.4,91.189.91.39,80.0,policy_indicator


: 

In [ ]:
df["exploit_class"].value_counts().head(20)

exploit_class
unknown             1913052
tls_fingerprint      420465
dns_suspicious       184831
rce                  121828
malware                8407
policy_indicator       7120
bruteforce_auth          90
ids_policy_info          24
recon_scan                4
Name: count, dtype: int64

: 

In [ ]:
df[df["ids_signature"].notna()]["exploit_class"].value_counts().head(20)


exploit_class
tls_fingerprint     420465
dns_suspicious      184831
policy_indicator      7120
unknown                914
ids_policy_info         24
recon_scan               4
Name: count, dtype: int64

: 

In [ ]:
df[df["scenario"] == "shaw"]["exploit_class"].value_counts()

exploit_class
unknown             35717
tls_fingerprint     32128
malware              1291
policy_indicator     1032
rce                   378
dns_suspicious        228
bruteforce_auth         4
recon_scan              4
Name: count, dtype: int64

: 

In [ ]:
df[df["scenario"] == "shaw"][["srcip","dstip"]].dropna().shape

(33634, 2)

: 

In [ ]:
df["ids_category"].value_counts().sort_index()

ids_category
(null)                                  3
Attempted User Privilege Gain          24
Generic Protocol Command Decode    421175
Misc activity                           4
Not Suspicious Traffic               6844
Potentially Bad Traffic            185029
Web Application Attack                276
Name: count, dtype: int64

: 

Check start and end times per scenario

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

for scenario, df_s in df.groupby("scenario"):
    t_min = df_s["timestamp"].min()
    t_max = df_s["timestamp"].max()
    delta = t_max - t_min

    days = delta.days
    hours = delta.total_seconds() / 3600
    minutes = delta.total_seconds() / 60

    print(
        f"{scenario:16s} | "
        f"start={t_min} | "
        f"end={t_max} | "
        f"Δ={days} days ({hours:.2f} h, {minutes:.0f} min)"
    )